# 03 · Race laps, weather and race control from TracingInsights (free)

Since October 2026, F1's timing APIs need an F1 TV login, so FastF1 can't download race lap data without a subscription (notebook 02). [TracingInsights](https://github.com/TracingInsights) published the same FastF1-derived data on GitHub, one repo per season, under the Apache-2.0 licence.

For each race this saves three files to Google Drive: **laps** (every lap of every driver, with tyre compound and age, stint, track status for safety car/VSC, pit in/out, accuracy flags and weather per lap), **weather** and **race control messages**.

Only those files are downloaded; the repos' telemetry, radio and cache folders are skipped. No rate limits apply, so one run covers all seasons. The repos are cloned to Colab's temporary disk, and only the parquet files go to Drive.

In [1]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive
try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Mounted at /content/drive
Cloning into '/content/f1-race-sim'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 74 (delta 34), reused 52 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 44.89 KiB | 2.64 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/f1-race-sim
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [2]:
# @title Download (one run is enough)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}

from f1sim.paths import get_paths
from f1sim.ingest import quiet_fastf1_logs
from f1sim.ingest_tracing import ingest_tracing_seasons

quiet_fastf1_logs(True)
paths = get_paths()
summary = ingest_tracing_seasons(range(START_YEAR, END_YEAR + 1), paths)

--- 2018 ---
saved 21 races
--- 2019 ---
saved 21 races
--- 2020 ---
saved 17 races
--- 2021 ---
saved 22 races
--- 2022 ---
saved 22 races
--- 2023 ---
saved 22 races
--- 2024 ---
saved 24 races
--- 2025 ---
saved 24 races
Saved this run:     173
Already on disk:    0
Skipped (future):   0
Failed:             0
Saved with warnings: 1
  - 2018 R01 Australian Grand Prix: no weather file, no rcm file


In [3]:
# @title Coverage per season
from f1sim.ingest_tracing import load_race_ti

laps = load_race_ti("laps", paths)
weather = load_race_ti("weather", paths)

per_race = laps.groupby(["Year", "Round", "Event"]).agg(
    Laps=("lap", "size"), Drivers=("drv", "nunique"),
    NoCompound=("compound", lambda s: int(s.isna().sum())),
).reset_index()
rain = weather.groupby(["Year", "Round"])["wR"].any().rename("AnyRain")
per_race = per_race.join(rain, on=["Year", "Round"])
per_race["AnyRain"] = per_race["AnyRain"].astype("boolean")
per_race["WeatherMissing"] = per_race["AnyRain"].isna().astype(int)

per_race.groupby("Year").agg(
    Races=("Round", "size"), Laps=("Laps", "sum"),
    MinDrivers=("Drivers", "min"), LapsNoCompound=("NoCompound", "sum"),
    RacesWithRain=("AnyRain", "sum"),
    WeatherMissing=("WeatherMissing", "sum"),
)

,Races,Laps,MinDrivers,LapsNoCompound,RacesWithRain,WeatherMissing
Year,,,,,,
2018,21,22342,19,382,8,1
2019,21,23676,20,0,3,0
2020,17,18436,20,0,4,0
2021,22,23758,19,0,6,0
2022,22,23577,18,334,6,0
2023,22,24422,19,35,5,0
2024,24,26606,19,0,5,0
2025,24,26692,19,503,6,0


In [4]:
# @title Tyre compounds per season
laps.groupby("Year")["compound"].value_counts(dropna=False).unstack(fill_value=0)

compound,HARD,HYPERSOFT,INTERMEDIATE,MEDIUM,SOFT,SUPERSOFT,ULTRASOFT,UNKNOWN,WET,<NA>
Year,,,,,,,,,,
2018,47,887,32,2970,7669,6033,4319,0,3,382
2019,7667,0,685,9159,6088,0,0,0,77,0
2020,6183,0,973,6946,4174,0,0,0,160,0
2021,10481,0,1730,8532,2923,0,0,60,32,0
2022,8035,0,1446,9204,4182,0,0,0,376,334
2023,11827,0,724,8312,3476,0,0,0,48,35
2024,13652,0,2178,8941,1798,0,0,0,37,0
2025,10843,0,1597,10444,3305,0,0,0,0,503


In [5]:
# @title Races where rain was recorded at any point
per_race[per_race["AnyRain"] == True][["Year", "Round", "Event", "Laps"]]

,Year,Round,Event,Laps
2,2018,3,Chinese Grand Prix,1115
4,2018,5,Spanish Grand Prix,1021
5,2018,6,Monaco Grand Prix,1516
7,2018,8,French Grand Prix,920
10,2018,11,German Grand Prix,1252
13,2018,14,Italian Grand Prix,928
15,2018,16,Russian Grand Prix,948
20,2018,21,Abu Dhabi Grand Prix,943
26,2019,6,Monaco Grand Prix,1490
31,2019,11,German Grand Prix,1059


In [6]:
# @title Drive space used
!du -sh "{paths.race_ti_dir}"

23M	/content/drive/MyDrive/f1sim_data/raw/race_tracinginsights


In [7]:
#laps with no compound
miss = laps[laps["compound"].isna() | (laps["compound"] == "UNKNOWN")]
miss.groupby(["Year", "Round", "Event"]).agg(
    Laps=("lap", "size"),
    Drivers=("drv", "nunique"),
    LapRange=("lap", lambda s: f"{s.min()}–{s.max()}"),
).reset_index()


,Year,Round,Event,Laps,Drivers,LapRange
0,2018,1,Australian Grand Prix,21,20,1–2
1,2018,2,Bahrain Grand Prix,23,20,1–35
2,2018,3,Chinese Grand Prix,21,20,1–2
3,2018,4,Azerbaijan Grand Prix,19,18,1–2
4,2018,5,Spanish Grand Prix,18,17,1–2
5,2018,6,Monaco Grand Prix,21,20,1–2
6,2018,7,Canadian Grand Prix,19,18,1–2
7,2018,8,French Grand Prix,19,18,1–2
8,2018,9,Austrian Grand Prix,21,20,1–2
9,2018,10,British Grand Prix,21,20,1–2
